# Experiment F (robustness run) — 3 seed draws + prevalence sensitivity

The 70-label frozen-feature probe transfers to field photographs (AUC 0.96–0.97), but the distilled
MobileNetV3 student does not (AUC 0.64–0.83). Here the probe pseudo-labels field photos and the student is
retrained, following the method of §III-F unchanged.

| Student | Training data | Extra expert labels |
|---|---|---|
| S0 | TeaLeafDiseaseBD train, pseudo-labelled (as published) | 0 |
| S1 | + field-train photos pseudo-labelled by the 70-label probe | **0** |
| S2 | + field-train photos pseudo-labelled by a probe fit on 10 field labels per class | 40 |

Field data: Chowdhury & Arman 2025, doi:10.17632/3x42rbj8yv.1 — four Habiganj gardens **not** among the eleven
TeaLeafDiseaseBD gardens, April–June 2025, iPhone, natural light. Split 50/50 with near-duplicate groups
(cosine ≥ 0.95) kept together, stratified by class. Students see field-train only; all field numbers are on
field-test. Inputs are phone-feasible: whole frame and a centre crop (no SAM). Every student is also scored
on the TeaLeafDiseaseBD test split (no-regression check). Thresholds: TeaLeafDiseaseBD-test 95% sensitivity
(fixed) and the label-free prevalence rule. This robustness run trains S0/S1/S2 for seed draws 0, 1, 2 and S1 with the field prevalence set to 16% and 36%
(11 students, ~80–90 min on a T4), saves all per-photo scores, and summarises mean ± SD and paired differences.

In [1]:
# CELL 0 — CONFIG
DRIVE      = '/content/drive/MyDrive'
TEA_DRIVE  = f'{DRIVE}/Tea Leaf Dataset'                              # one folder per class
STUDY      = f'{DRIVE}/TeaLeaf_DINOv2_Study'
RES        = f'{STUDY}/results_revision_v1'
FEATS      = f'{RES}/features_dinov1_crop.npz'
SPLIT      = f'{RES}/_split.json'
EJSON      = f'{RES}/E_distillation_5rep.json'
ESCORES    = f'{RES}/E_scores_5rep.npz'
DEPLOY     = f'{RES}/deploy'
LOCAL      = '/content/tea_leaf_local'
CACHE      = '/content/crop_cache'
SEED, N_SHOT, CROP_PAD, IMG = 42, 10, 0.06, 224
REPS       = []           # Experiment F trains its own students (below)
EXPORT_REP = 1            # which trained student to export (1 = draw closest to the 5-seed mean)
EPOCHS, BS, LR = 30, 64, 1e-3

# ---- Experiment F ----
FIELD_ZIP_OR_DIR = f'{DRIVE}/Tea Field Dataset'
FIELD_FEAT_WHOLE = f'{RES}/field_test/feat_Field_whole.npz'   # from field_test.ipynb (alignment + split groups)
OUTF       = f'{RES}/experiment_F'
F_REPS     = [0, 1, 2]    # seed draws of the 70 TeaLeaf labels
PI_SENS    = [0.16, 0.36] # S1 retrained with the field healthy prevalence misestimated by -/+10 points
PI_REP     = 1            # seed draw used for the prevalence-sensitivity runs
CENTRE_FRAC= 0.60         # centre crop = central 60% square of the frame (phone-feasible framing guide)
DUP        = 0.95         # near-duplicate grouping for the split and the bootstrap
FIELD_PI   = 0.26         # healthy prevalence of the field set (label-free rule; a field walk would estimate it)
TARGET_SENS= 0.95
TRAIN_S0, TRAIN_S2 = True, True


In [2]:
# CELL 1 — setup and local copy of the dataset
from google.colab import drive; drive.mount('/content/drive')
!pip -q install onnx onnxruntime timm
import os, io, re, json, time, glob, shutil, hashlib, random, warnings
import numpy as np, torch, torch.nn as nn, torch.optim as optim
import torchvision.models as tvm, torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy import ndimage
from scipy.stats import spearmanr
import cv2
warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device', device)
os.makedirs(DEPLOY, exist_ok=True); os.makedirs(CACHE, exist_ok=True)
IMG_EXT = ('.jpg', '.jpeg', '.png')

def _cp(t):
    s, d = t
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True); shutil.copyfile(s, d)
t0 = time.time()
tasks = [(os.path.join(r, f), os.path.join(LOCAL, os.path.relpath(r, TEA_DRIVE), f))
         for r, _, fs in os.walk(TEA_DRIVE) for f in fs if f.lower().endswith(IMG_EXT)]
with ThreadPoolExecutor(32) as ex: list(ex.map(_cp, tasks))
print('local copy: %d files in %.0fs' % (len(tasks), time.time() - t0))


Mounted at /content/drive
device cuda
local copy: 13084 files in 306s


In [3]:
# CELL 2 — records in the original order: classes alphabetical, files sorted within class
classes = sorted(d for d in os.listdir(LOCAL) if os.path.isdir(os.path.join(LOCAL, d)))
HEALTHY = classes.index(next(c for c in classes if 'healthy' in c.lower()))
RECS = [dict(rel=f'{c}/{f}', path=os.path.join(LOCAL, c, f), y=k, d=int(k != HEALTHY))
        for k, c in enumerate(classes) for f in sorted(os.listdir(os.path.join(LOCAL, c)))
        if f.lower().endswith(IMG_EXT)]
print('classes:', classes, '| healthy index', HEALTHY)
print('records:', len(RECS))
assert len(classes) == 7 and len(RECS) == 13084, 'dataset differs from the published release'


classes: ['Algal leaf spot', 'Brown Blight', 'Gray Blight', 'Healthy', 'Helopelties', 'Red Rust', 'Red Spider'] | healthy index 3
records: 13084


In [4]:
# CELL 3 — restore the group-aware split from _split.json, check labels against the cached features
SPL = json.load(open(SPLIT)); F = np.load(FEATS)
test_rel = ['/'.join(p.split('/')[-2:]) for p in SPL['test_paths']]
by_rel = {r['rel']: r for r in RECS}
missing = [p for p in test_rel if p not in by_rel]
assert not missing, f'{len(missing)} test files not found, e.g. {missing[:3]}'
TEST = [by_rel[p] for p in test_rel]                       # exact published test order
tset = set(test_rel)
TRAIN = [r for r in RECS if r['rel'] not in tset]          # record order (original convention)
ytr, dtr = np.array([r['y'] for r in TRAIN]), np.array([r['d'] for r in TRAIN])
yte, dte = np.array([r['y'] for r in TEST]), np.array([r['d'] for r in TEST])
print('train %d | test %d | test diseased %.4f' % (len(TRAIN), len(TEST), dte.mean()))
assert (len(TRAIN), len(TEST)) == (SPL['n_train'], SPL['n_test'])
assert (yte == F['yte']).all() and (dte == F['dte']).all(), 'test labels differ from cached features'
TRAIN_ORDER_OK = len(ytr) == len(F['ytr']) and (ytr == F['ytr']).all()
print('train label sequence matches cache:', TRAIN_ORDER_OK, '(order within class is checked in Cell 6)')


train 9155 | test 3929 | test diseased 0.8636
train label sequence matches cache: True (order within class is checked in Cell 6)


In [5]:
# CELL 4 — segmentation and leaf crop
# Prefer the ORIGINAL modules from Drive (shape_confound.leaf_mask, feature_shape_control.crop_to_leaf);
# fall back to the paper's recipe: Otsu on HSV V|S, largest component, fill holes, bbox + 6% pad, 224x224.
import sys
for m in ('shape_confound.py', 'feature_shape_control.py'):
    hit = glob.glob(f'{STUDY}/**/{m}', recursive=True)
    if hit: shutil.copy(hit[0], f'/content/{m}')
sys.path.insert(0, '/content')
try:
    import shape_confound as _sc, feature_shape_control as _fsc
    _ORIG = True
except Exception as e:
    _ORIG = False; print('original modules not found (%s) -> fallback recipe' % e)

def _fb_mask(rgb):
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    _, mv = cv2.threshold(hsv[..., 2], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, ms = cv2.threshold(hsv[..., 1], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = (mv > 0) | (ms > 0); lab, n = ndimage.label(m)
    if n == 0: return np.ones(m.shape, bool)
    big = 1 + int(np.argmax(ndimage.sum(m, lab, range(1, n + 1))))
    return ndimage.binary_fill_holes(lab == big)

def _fb_crop(rgb, m, pad=CROP_PAD, size=IMG):
    ys, xs = np.where(m); H, W = m.shape
    x0, x1, y0, y1 = xs.min(), xs.max() + 1, ys.min(), ys.max() + 1
    px, py = pad * (x1 - x0), pad * (y1 - y0)
    x0, y0, x1, y1 = int(max(0, x0 - px)), int(max(0, y0 - py)), int(min(W, x1 + px)), int(min(H, y1 + py))
    return Image.fromarray(rgb[y0:y1, x0:x1]).resize((size, size), Image.BILINEAR)

def leaf_crop(path):
    rgb = np.array(Image.open(path).convert('RGB'))
    if _ORIG:
        try:
            try: m = _sc.leaf_mask(rgb)
            except Exception: m = _sc.leaf_mask(Image.fromarray(rgb))
            im = _fsc.crop_to_leaf(rgb, np.asarray(m).astype(bool))
            im = im if isinstance(im, Image.Image) else Image.fromarray(np.asarray(im).astype(np.uint8))
            return im.convert('RGB').resize((IMG, IMG)) if im.size != (IMG, IMG) else im.convert('RGB')
        except Exception:
            pass
    return _fb_crop(rgb, _fb_mask(rgb))
print('crop function:', 'ORIGINAL modules' if _ORIG else 'fallback recipe')


crop function: ORIGINAL modules


In [6]:
# CELL 5 — crop cache (JPEG q95, same as the revision_v1 run), parallel
def cache_path(rec):
    return os.path.join(CACHE, hashlib.md5(rec['rel'].encode()).hexdigest() + '.jpg')
def _build(rec):
    cp = cache_path(rec)
    if not os.path.exists(cp):
        try: leaf_crop(rec['path']).save(cp, quality=95)
        except Exception as e: return rec['rel'], str(e)
    return None
t0 = time.time()
with ProcessPoolExecutor(os.cpu_count()) as ex:
    fails = [f for f in ex.map(_build, RECS, chunksize=32) if f]
print('crop cache: %d images, %d failures, %.0fs' % (len(RECS), len(fails), time.time() - t0))
assert not fails, fails[:5]


crop cache: 13084 images, 0 failures, 409s


In [7]:
# Meta's server currently fails certificate verification in Colab; integrity is checked below (features must
# reproduce the cached training features). Normal verification is restored afterwards.
import ssl; _ssl_default = ssl._create_default_https_context; ssl._create_default_https_context = ssl._create_unverified_context
# CELL 6 — DINO ViT-S/16 features: use the cache only if our crops reproduce it
TO_T = T.Compose([T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
dino = torch.hub.load('facebookresearch/dino:main', 'dino_vits16').to(device).eval()
@torch.no_grad()
def feats(recs, bs=128):
    out = []
    for i in range(0, len(recs), bs):
        x = torch.stack([TO_T(Image.open(cache_path(r)).convert('RGB')) for r in recs[i:i + bs]]).to(device)
        out.append(torch.nn.functional.normalize(dino(x).float(), dim=1).cpu().numpy())
    return np.concatenate(out)
rng = np.random.default_rng(0)
it, ie = rng.choice(len(TRAIN), 64, replace=False), rng.choice(len(TEST), 64, replace=False)
ct = (feats([TRAIN[i] for i in it]) * F['Xtr'][it]).sum(1) if TRAIN_ORDER_OK else np.zeros(1)
ce = (feats([TEST[i] for i in ie]) * F['Xte'][ie]).sum(1)
print('cosine to cache: train median %.4f | test median %.4f' % (np.median(ct), np.median(ce)))
USE_CACHE = TRAIN_ORDER_OK and np.median(ct) >= 0.99 and np.median(ce) >= 0.99
if USE_CACHE:
    Xtr, Xte = F['Xtr'], F['Xte']; print('-> using cached features (exact reproduction of the published run)')
else:
    print('-> cache does not align with these crops: re-extracting all features (numbers will differ slightly)')
    Xtr, Xte = feats(TRAIN), feats(TEST)
    np.savez(f'{RES}/features_dinov1_crop_rebuilt.npz', Xtr=Xtr, ytr=ytr, dtr=dtr, Xte=Xte, yte=yte, dte=dte)
ssl._create_default_https_context = _ssl_default   # keep DINO loaded for the field centre crops


Downloading: "https://github.com/facebookresearch/dino/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/dino/dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dino_deitsmall16_pretrain.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 391MB/s]


cosine to cache: train median 0.9962 | test median 0.9965
-> using cached features (exact reproduction of the published run)


In [8]:
# CELL 7 — 70-label probe (multinomial logistic, C=1.0) and the binary score; reproduce published AUCs
J = json.load(open(EJSON))
def draw(rep, n=N_SHOT):
    r = np.random.default_rng(SEED * 1000 + rep)
    return np.concatenate([r.choice(np.where(ytr == c)[0], n, replace=False) for c in range(len(classes))])
def fit_probe(X, y, idx):
    return LogisticRegression(C=1.0, max_iter=2000).fit(X[idx], y[idx])
def binary_score(probe, X):
    return 1.0 - probe.predict_proba(X)[:, HEALTHY]
pa = [roc_auc_score(dte, binary_score(fit_probe(Xtr, ytr, draw(r)), Xte)) for r in range(5)]
print('probe AUC per draw:', np.round(pa, 4)); print('published         :', np.round(J['probe_auc_per_rep'], 4))
if USE_CACHE:
    assert np.allclose(pa, J['probe_auc_per_rep'], atol=1e-3), 'probe did not reproduce: stop'
    print('-> exact reproduction')


probe AUC per draw: [0.9791 0.9858 0.9888 0.9836 0.9893]
published         : [0.9791 0.9858 0.9888 0.9836 0.9893]
-> exact reproduction


In [9]:
# CELL 8 — train the student(s): prevalence-quantile hard pseudo-labels, MobileNetV3-Small (Sec III-F)
class ArrDS(Dataset):
    def __init__(self, recs, labels, train):
        self.r, self.l, self.train = recs, labels, train
        self.aug = T.Compose([T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                              T.RandomRotation(20), T.ColorJitter(.2, .2, .2)])
    def __len__(self): return len(self.r)
    def __getitem__(self, i):
        im = Image.open(cache_path(self.r[i])).convert('RGB')
        if self.train: im = self.aug(im)
        return TO_T(im), int(self.l[i])

def train_student(recs, labels, seed):
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    m = tvm.mobilenet_v3_small(weights=tvm.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    m.classifier[3] = nn.Linear(m.classifier[3].in_features, 2); m = m.to(device)
    dl = DataLoader(ArrDS(recs, labels, True), batch_size=BS, shuffle=True, num_workers=4)
    opt = optim.Adam(m.parameters(), lr=LR); sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    lf = nn.CrossEntropyLoss()
    for ep in range(EPOCHS):
        m.train()
        for x, y in dl:
            opt.zero_grad(); lf(m(x.to(device)), y.to(device)).backward(); opt.step()
        sch.step()
    return m.eval()

@torch.no_grad()
def student_scores(m, recs):
    dl = DataLoader(ArrDS(recs, np.zeros(len(recs)), False), batch_size=128, num_workers=4)
    return np.concatenate([torch.softmax(m(x.to(device)), 1)[:, 1].cpu().numpy() for x, _ in dl])

E5 = np.load(ESCORES)['probe_distil']; pub = [roc_auc_score(dte, e) for e in E5]
STUDENTS, LOG = {}, []
for rep in REPS:
    idx = draw(rep); probe = fit_probe(Xtr, ytr, idx); s_tr = binary_score(probe, Xtr)
    pi = 1.0 - float(dtr.mean()); tau = float(np.quantile(s_tr, pi))
    pseudo = (s_tr > tau).astype(int); pseudo[idx] = dtr[idx]
    t0 = time.time(); m = train_student(TRAIN, pseudo, seed=1000 + rep); s = student_scores(m, TEST)
    a = roc_auc_score(dte, s); rho = spearmanr(s, E5[rep])[0]
    torch.save(m.state_dict(), f'{DEPLOY}/student_rep{rep}.pt'); np.save(f'{DEPLOY}/student_scores_rep{rep}.npy', s)
    STUDENTS[rep] = m
    LOG.append(dict(rep=rep, tau=tau, pseudo_acc=float((pseudo == dtr).mean()), auc=float(a),
                    published_auc=float(pub[rep]), spearman_vs_published=float(rho), minutes=(time.time() - t0) / 60))
    print('rep %d | tau %.4f | pseudo acc %.4f | AUC %.4f (published %.4f) | Spearman %.3f | %.1f min'
          % (rep, tau, LOG[-1]['pseudo_acc'], a, pub[rep], rho, LOG[-1]['minutes']))
lo, hi = min(pub), max(pub)
for L in LOG:
    L['within_published_spread'] = bool(lo - 0.005 <= L['auc'] <= hi + 0.005)
    if not L['within_published_spread']: print('WARNING rep %d outside the published spread %.4f-%.4f' % (L['rep'], lo, hi))
json.dump(LOG, open(f'{DEPLOY}/students.json', 'w'), indent=2)


In [10]:
# CELL F1 — field records (same order as field_test.ipynb), whole-frame and centre crops into the cache
import zipfile
from collections import Counter
os.makedirs(OUTF, exist_ok=True)
FIELD_LOCAL = '/content/field_local'
if not os.path.exists(FIELD_LOCAL):
    zips = glob.glob(f'{FIELD_ZIP_OR_DIR}/**/*.zip', recursive=True)
    if zips:
        for z in zips: zipfile.ZipFile(z).extractall(FIELD_LOCAL)
    else: shutil.copytree(FIELD_ZIP_OR_DIR, FIELD_LOCAL)
    for z in glob.glob(f'{FIELD_LOCAL}/**/*.zip', recursive=True): zipfile.ZipFile(z).extractall(os.path.dirname(z))
def field_class(p):
    d = os.path.basename(os.path.dirname(p)).lower().replace('_', ' ')
    return ('Healthy' if 'healthy' in d else 'Red Rust' if 'rust' in d else 'Helopeltis' if 'helopelt' in d
            else 'Blight' if 'blight' in d else None)
files = sorted(p for p in glob.glob(f'{FIELD_LOCAL}/**/*', recursive=True)
               if p.lower().endswith(IMG_EXT) and '__MACOSX' not in p and field_class(p))
FREC = []
for p in files:
    c = field_class(p); rel = f'FIELD/{c}/{os.path.basename(p)}'
    FREC.append(dict(path=p, cls=c, d=int(c != 'Healthy'),
                     whole=dict(rel=rel + '#whole'), centre=dict(rel=rel + '#centre')))
fy = np.array([r['d'] for r in FREC]); fcls = np.array([r['cls'] for r in FREC])
W = np.load(FIELD_FEAT_WHOLE, allow_pickle=True)
assert len(fy) == len(W['y']) and (fy == W['y']).all() and (fcls == W['cls']).all(), \
    'field records do not align with feat_Field_whole.npz (same download/order as field_test.ipynb needed)'
Xf_whole = W['X']
print('field:', Counter(fcls), '| aligned with field_test features')
def _fcrops(r):
    rgb = np.array(Image.open(r['path']).convert('RGB')); H, Wd = rgb.shape[:2]
    out = {}
    for key, box in (('whole', (0, 0, Wd, H)),
                     ('centre', (Wd * (1 - CENTRE_FRAC) / 2, H * (1 - CENTRE_FRAC) / 2,
                                 Wd * (1 + CENTRE_FRAC) / 2, H * (1 + CENTRE_FRAC) / 2))):
        cp = cache_path(r[key])
        if not os.path.exists(cp):
            x0, y0, x1, y1 = map(int, box)
            Image.fromarray(rgb[y0:y1, x0:x1]).resize((IMG, IMG), Image.BILINEAR).save(cp, quality=95)
    return True
t0 = time.time()
with ThreadPoolExecutor(8) as ex: list(ex.map(_fcrops, FREC))
print('field crops cached (%.0fs)' % (time.time() - t0))


field: Counter({np.str_('Helopeltis'): 675, np.str_('Healthy'): 650, np.str_('Red Rust'): 620, np.str_('Blight'): 555}) | aligned with field_test features
field crops cached (116s)


In [11]:
# CELL F2 — centre-crop DINO features and zero-shot probe check (whole-frame features come from field_test)
Xf_centre = feats([r['centre'] for r in FREC])
del dino; torch.cuda.empty_cache()
print('centre crops: %d features' % len(Xf_centre))
ZS = {}
for view, X in (('whole', Xf_whole), ('centre', Xf_centre)):
    a = [roc_auc_score(fy, binary_score(fit_probe(Xtr, ytr, draw(r)), X)) for r in range(5)]
    ZS[view] = (float(np.mean(a)), float(np.std(a, ddof=1)))
    print('zero-shot probe, field %-6s AUC %.3f ± %.3f (5 draws)' % (view, *ZS[view]))


centre crops: 2500 features
zero-shot probe, field whole  AUC 0.962 ± 0.017 (5 draws)
zero-shot probe, field centre AUC 0.973 ± 0.009 (5 draws)


In [12]:
# CELL F3 — field train/test split: near-duplicate groups kept together, stratified by class
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
grp = connected_components(csr_matrix(Xf_whole @ Xf_whole.T >= DUP), directed=False)[1]
rng = np.random.default_rng(2026)
is_tr = np.zeros(len(FREC), bool)
for c in np.unique(fcls):
    gs = np.unique(grp[fcls == c]); rng.shuffle(gs)
    cum, half = 0, (fcls == c).sum() / 2
    for g in gs:
        if cum >= half: break
        is_tr[grp == g] = True; cum += (grp == g).sum()
assert not set(grp[is_tr]) & set(grp[~is_tr]), 'a near-duplicate group crosses the split'
F_TR, F_TE = np.where(is_tr)[0], np.where(~is_tr)[0]
print('field-train %d | field-test %d | groups crossing the split: 0' % (len(F_TR), len(F_TE)))
print('field-test classes:', Counter(fcls[F_TE]))
json.dump(dict(dup=DUP, train=[FREC[i]['whole']['rel'][:-6] for i in F_TR], test=[FREC[i]['whole']['rel'][:-6] for i in F_TE]),
          open(f'{OUTF}/field_split.json', 'w'), indent=1)


field-train 1252 | field-test 1248 | groups crossing the split: 0
field-test classes: Counter({np.str_('Helopeltis'): 335, np.str_('Healthy'): 325, np.str_('Red Rust'): 311, np.str_('Blight'): 277})


In [13]:
# CELL F4 — train S0 / S1 / S2 for every seed draw, plus S1 under misestimated prevalence; save all scores
from sklearn.metrics import roc_auc_score
def thr_at_sens(s, d, t=TARGET_SENS): return float(np.sort(s[d == 1])[int(np.floor((1 - t) * (d == 1).sum()))])
def sens_spec(s, d, thr): f = s >= thr; return float(f[d == 1].mean()), float((~f)[d == 0].mean())
def group_boot_auc(s, d, g, B=1000, seed=0):
    r = np.random.default_rng(seed); ug = np.unique(g); idx = {k: np.where(g == k)[0] for k in ug}; out = []
    for _ in range(B):
        sel = np.concatenate([idx[k] for k in r.choice(ug, len(ug), replace=True)])
        if len(np.unique(d[sel])) == 2: out.append(roc_auc_score(d[sel], s[sel]))
    return [float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))]
FIELD_TR_RECS = [FREC[i][v] for i in F_TR for v in ('whole', 'centre')]
ct = np.unique(fcls); dF = fy[F_TE]
RES_F = {'zero_shot_probe': ZS, 'split': dict(n_train=int(len(F_TR)), n_test=int(len(F_TE)))}
SCORES, STU = {'tealeaf_test_d': dte, 'field_test_d': dF, 'field_test_cls': fcls[F_TE], 'field_test_group': grp[F_TE]}, {}

def run_student(name, rep, recs, labels, extra):
    t0 = time.time(); m = train_student(recs, labels, seed=1000 + rep)
    s_te = student_scores(m, TEST); thr = thr_at_sens(s_te, dte)
    row = dict(rep=rep, minutes=round((time.time() - t0) / 60, 1), **extra,
               tealeaf_test=dict(auc=float(roc_auc_score(dte, s_te)), sens_spec_fixed=sens_spec(s_te, dte, thr)))
    SCORES[f'{name}_rep{rep}_tealeaf'] = s_te
    for view in ('whole', 'centre'):
        sv = student_scores(m, [FREC[i][view] for i in F_TE]); SCORES[f'{name}_rep{rep}_field_{view}'] = sv
        row[f'field_{view}'] = dict(auc=float(roc_auc_score(dF, sv)), auc_ci95_group=group_boot_auc(sv, dF, grp[F_TE]),
                                    sens_spec_fixed_tealeaf_thr=sens_spec(sv, dF, thr),
                                    sens_spec_label_free=sens_spec(sv, dF, float(np.quantile(sv, FIELD_PI))),
                                    flag_rate_by_class={c: float((sv[fcls[F_TE] == c] >= thr).mean()) for c in ct})
    torch.save(m.state_dict(), f'{OUTF}/student_{name}_rep{rep}.pt'); STU[(name, rep)] = m.cpu()
    RES_F[f'{name}_rep{rep}'] = row
    print('  %-10s rep %d: TeaLeaf %.3f | field whole %.3f [%.3f, %.3f] centre %.3f [%.3f, %.3f] | spec@fixed %.2f / %.2f | %.1f min'
          % (name, rep, row['tealeaf_test']['auc'], row['field_whole']['auc'], *row['field_whole']['auc_ci95_group'],
             row['field_centre']['auc'], *row['field_centre']['auc_ci95_group'],
             row['field_whole']['sens_spec_fixed_tealeaf_thr'][1], row['field_centre']['sens_spec_fixed_tealeaf_thr'][1], row['minutes']))
    torch.cuda.empty_cache(); json.dump(RES_F, open(f'{OUTF}/experiment_F_robustness.json', 'w'), indent=2, default=float)
    np.savez(f'{OUTF}/experiment_F_scores.npz', **SCORES)      # saved after every student, so a crash loses nothing

def field_pl(probe, pi):
    sf = 0.5 * (binary_score(probe, Xf_whole[F_TR]) + binary_score(probe, Xf_centre[F_TR]))
    return (sf > np.quantile(sf, pi)).astype(int)

for rep in F_REPS:
    idx = draw(rep); probe = fit_probe(Xtr, ytr, idx); s_tr = binary_score(probe, Xtr)
    tau = float(np.quantile(s_tr, 1.0 - float(dtr.mean()))); pl_tea = (s_tr > tau).astype(int); pl_tea[idx] = dtr[idx]
    pl_s1 = field_pl(probe, FIELD_PI)
    r2 = np.random.default_rng(5000 + rep)
    lab = np.concatenate([r2.choice(F_TR[fcls[F_TR] == c], 10, replace=False) for c in ct])
    p2 = LogisticRegression(C=1.0, max_iter=2000).fit(np.vstack([Xf_whole[lab], Xf_centre[lab]]), np.concatenate([fcls[lab]] * 2))
    hk = list(p2.classes_).index('Healthy')
    s2 = 0.5 * ((1 - p2.predict_proba(Xf_whole[F_TR])[:, hk]) + (1 - p2.predict_proba(Xf_centre[F_TR])[:, hk]))
    pl_s2 = (s2 > np.quantile(s2, FIELD_PI)).astype(int); m_lab = np.isin(F_TR, lab); pl_s2[m_lab] = fy[F_TR[m_lab]]
    acc1, acc2 = float((pl_s1 == fy[F_TR]).mean()), float((pl_s2 == fy[F_TR]).mean())
    print('rep %d | field pseudo-label accuracy: S1 %.3f | S2 %.3f' % (rep, acc1, acc2))
    if TRAIN_S0: run_student('S0', rep, TRAIN, pl_tea, dict(expert_field_labels=0, field_pi=None, field_pseudo_acc=None))
    run_student('S1', rep, TRAIN + FIELD_TR_RECS, np.concatenate([pl_tea, np.repeat(pl_s1, 2)]),
                dict(expert_field_labels=0, field_pi=FIELD_PI, field_pseudo_acc=acc1))
    if TRAIN_S2: run_student('S2', rep, TRAIN + FIELD_TR_RECS, np.concatenate([pl_tea, np.repeat(pl_s2, 2)]),
                             dict(expert_field_labels=40, field_pi=FIELD_PI, field_pseudo_acc=acc2))
    if rep == PI_REP:
        for pi in PI_SENS:
            pl = field_pl(probe, pi); acc = float((pl == fy[F_TR]).mean())
            print('rep %d | S1 with field prevalence %.2f: pseudo-label accuracy %.3f' % (rep, pi, acc))
            run_student('S1_pi%02d' % round(100 * pi), rep, TRAIN + FIELD_TR_RECS, np.concatenate([pl_tea, np.repeat(pl, 2)]),
                        dict(expert_field_labels=0, field_pi=pi, field_pseudo_acc=acc))
print('saved ->', f'{OUTF}/experiment_F_robustness.json and experiment_F_scores.npz')


rep 0 | field pseudo-label accuracy: S1 0.898 | S2 0.974
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 215MB/s]


  S0         rep 0: TeaLeaf 0.980 | field whole 0.581 [0.528, 0.631] centre 0.860 [0.819, 0.895] | spec@fixed 0.02 / 0.38 | 5.0 min
  S1         rep 0: TeaLeaf 0.978 | field whole 0.956 [0.936, 0.973] centre 0.954 [0.929, 0.973] | spec@fixed 0.94 / 0.94 | 6.5 min
  S2         rep 0: TeaLeaf 0.972 | field whole 0.990 [0.977, 0.998] centre 0.984 [0.968, 0.997] | spec@fixed 0.97 / 0.97 | 6.6 min
rep 1 | field pseudo-label accuracy: S1 0.929 | S2 0.966
  S0         rep 1: TeaLeaf 0.986 | field whole 0.691 [0.645, 0.732] centre 0.891 [0.858, 0.920] | spec@fixed 0.07 / 0.37 | 5.4 min
  S1         rep 1: TeaLeaf 0.983 | field whole 0.975 [0.958, 0.987] centre 0.978 [0.962, 0.990] | spec@fixed 0.96 / 0.95 | 6.8 min
  S2         rep 1: TeaLeaf 0.984 | field whole 0.987 [0.972, 0.996] centre 0.990 [0.977, 0.997] | spec@fixed 0.97 / 0.97 | 6.8 min
rep 1 | S1 with field prevalence 0.16: pseudo-label accuracy 0.883
  S1_pi16    rep 1: TeaLeaf 0.983 | field whole 0.939 [0.915, 0.961] centre 0.941 [0

In [14]:
# CELL F5 — summary across seed draws: mean ± SD, paired S1 − S0 and S2 − S1
def g(k, rep, *path):
    v = RES_F[f'{k}_rep{rep}']
    for p in path: v = v[p]
    return v
SUM = {}
for k in ('S0', 'S1', 'S2'):
    reps = [r for r in F_REPS if f'{k}_rep{r}' in RES_F]
    if not reps: continue
    row = {}
    for name, path in (('tealeaf_auc', ('tealeaf_test', 'auc')), ('field_whole_auc', ('field_whole', 'auc')),
                       ('field_centre_auc', ('field_centre', 'auc'))):
        v = [g(k, r, *path) for r in reps]; row[name] = (float(np.mean(v)), float(np.std(v, ddof=1)) if len(v) > 1 else 0.0)
    for view in ('whole', 'centre'):
        for i, lab_ in ((0, 'sens'), (1, 'spec')):
            v = [g(k, r, f'field_{view}', 'sens_spec_fixed_tealeaf_thr')[i] for r in reps]
            row[f'field_{view}_{lab_}_fixed'] = (float(np.mean(v)), float(np.std(v, ddof=1)) if len(v) > 1 else 0.0)
            v = [g(k, r, f'field_{view}', 'sens_spec_label_free')[i] for r in reps]
            row[f'field_{view}_{lab_}_labelfree'] = (float(np.mean(v)), float(np.std(v, ddof=1)) if len(v) > 1 else 0.0)
    SUM[k] = row
fmt = lambda t: '%.3f ± %.3f' % t
print('%-4s %16s %16s %16s %16s %16s' % ('', 'TeaLeaf AUC', 'field whole AUC', 'field centre AUC', 'spec fixed (c)', 'spec label-free (c)'))
for k, row in SUM.items():
    print('%-4s %16s %16s %16s %16s %16s' % (k, fmt(row['tealeaf_auc']), fmt(row['field_whole_auc']), fmt(row['field_centre_auc']),
                                          fmt(row['field_centre_spec_fixed']), fmt(row['field_centre_spec_labelfree'])))
PAIRED = {}
for a, b in (('S1', 'S0'), ('S2', 'S1')):
    for view in ('whole', 'centre'):
        d = [g(a, r, f'field_{view}', 'auc') - g(b, r, f'field_{view}', 'auc') for r in F_REPS
             if f'{a}_rep{r}' in RES_F and f'{b}_rep{r}' in RES_F]
        if d: PAIRED[f'{a}-{b}_{view}'] = dict(per_rep=d, mean=float(np.mean(d)), min=float(np.min(d)))
for k, v in PAIRED.items(): print('paired %-16s mean %+.3f, per seed %s' % (k, v['mean'], np.round(v['per_rep'], 3)))
PI = {k: dict(field_pi=v['field_pi'], pseudo_acc=v['field_pseudo_acc'], whole=v['field_whole']['auc'], centre=v['field_centre']['auc'],
              tealeaf=v['tealeaf_test']['auc']) for k, v in RES_F.items() if isinstance(v, dict) and str(k).startswith('S1_pi')}
PI[f'S1_rep{PI_REP} (pi {FIELD_PI})'] = dict(field_pi=FIELD_PI, pseudo_acc=RES_F[f'S1_rep{PI_REP}']['field_pseudo_acc'],
                                             whole=RES_F[f'S1_rep{PI_REP}']['field_whole']['auc'],
                                             centre=RES_F[f'S1_rep{PI_REP}']['field_centre']['auc'],
                                             tealeaf=RES_F[f'S1_rep{PI_REP}']['tealeaf_test']['auc'])
for k, v in sorted(PI.items(), key=lambda kv: kv[1]['field_pi']):
    print('prevalence %.2f: pseudo-label acc %.3f | field AUC whole %.3f centre %.3f | TeaLeaf %.3f' %
          (v['field_pi'], v['pseudo_acc'], v['whole'], v['centre'], v['tealeaf']))
RES_F['_summary'] = dict(mean_sd=SUM, paired=PAIRED, prevalence_sensitivity=PI, reps=F_REPS)
json.dump(RES_F, open(f'{OUTF}/experiment_F_robustness.json', 'w'), indent=2, default=float)


          TeaLeaf AUC  field whole AUC field centre AUC   spec fixed (c) spec label-free (c)
S0      0.986 ± 0.006    0.719 ± 0.154    0.899 ± 0.043    0.495 ± 0.208    0.739 ± 0.056
S1      0.983 ± 0.005    0.972 ± 0.015    0.973 ± 0.018    0.957 ± 0.022    0.893 ± 0.040
S2      0.981 ± 0.008    0.989 ± 0.002    0.988 ± 0.003    0.976 ± 0.012    0.941 ± 0.012
paired S1-S0_whole      mean +0.253, per seed [0.375 0.284 0.099]
paired S1-S0_centre     mean +0.074, per seed [0.094 0.087 0.042]
paired S2-S1_whole      mean +0.017, per seed [0.034 0.012 0.005]
paired S2-S1_centre     mean +0.015, per seed [0.031 0.011 0.003]
prevalence 0.16: pseudo-label acc 0.883 | field AUC whole 0.939 centre 0.941 | TeaLeaf 0.983
prevalence 0.26: pseudo-label acc 0.929 | field AUC whole 0.975 centre 0.978 | TeaLeaf 0.983
prevalence 0.36: pseudo-label acc 0.898 | field AUC whole 0.983 centre 0.979 | TeaLeaf 0.985


In [15]:
# CELL F6 — export the S1 student closest to the 3-seed mean (field centre AUC) to ONNX and verify
import onnx, onnxruntime as ort, inspect
mu = SUM['S1']['field_centre_auc'][0]
rep = min(F_REPS, key=lambda r: abs(RES_F[f'S1_rep{r}']['field_centre']['auc'] - mu)); m = STU[('S1', rep)].eval()
onnx_path = f'{OUTF}/tea_screener_field_S1_rep{rep}.onnx'
kw = dict(opset_version=13, input_names=['input'], output_names=['logits'],
          dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}}, do_constant_folding=True)
if 'dynamo' in inspect.signature(torch.onnx.export).parameters: kw['dynamo'] = False
torch.onnx.export(m, torch.randn(1, 3, IMG, IMG), onnx_path, **kw)
om = onnx.load(onnx_path, load_external_data=True); onnx.save_model(om, onnx_path, save_as_external_data=False)
for extra in glob.glob(onnx_path + '.data'): os.remove(extra)
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
x = torch.stack([TO_T(Image.open(cache_path(FREC[i]['centre'])).convert('RGB')) for i in F_TE[:256]])
with torch.no_grad(): diff = float(np.abs(sess.run(None, {'input': x.numpy()})[0] - m(x).numpy()).max())
rep_onnx = dict(onnx=os.path.basename(onnx_path), rep=rep, field_centre_auc=RES_F[f'S1_rep{rep}']['field_centre']['auc'],
                field_whole_auc=RES_F[f'S1_rep{rep}']['field_whole']['auc'], tealeaf_auc=RES_F[f'S1_rep{rep}']['tealeaf_test']['auc'],
                size_mb=os.path.getsize(onnx_path) / 1e6, max_logit_diff=diff,
                sha256=hashlib.sha256(open(onnx_path, 'rb').read()).hexdigest(),
                preprocessing='whole frame or centre 60% square, 224x224, ImageNet mean/std', output='logits[:,1] = diseased')
json.dump(rep_onnx, open(f'{OUTF}/export_report_field_S1.json', 'w'), indent=2)
print(json.dumps(rep_onnx, indent=2)); assert diff < 1e-3 and rep_onnx['size_mb'] > 5
print('Send me experiment_F_robustness.json, experiment_F_scores.npz and export_report_field_S1.json')


{
  "onnx": "tea_screener_field_S1_rep1.onnx",
  "rep": 1,
  "field_centre_auc": 0.978404867072256,
  "field_whole_auc": 0.9749045753812818,
  "tealeaf_auc": 0.9828204798289719,
  "size_mb": 6.094012,
  "max_logit_diff": 4.1961669921875e-05,
  "sha256": "ab2522118ffb9a0cb74416cf2f1361c6b5deaa1f6f3be663bd0c04501a838d1f",
  "preprocessing": "whole frame or centre 60% square, 224x224, ImageNet mean/std",
  "output": "logits[:,1] = diseased"
}
Send me experiment_F_robustness.json, experiment_F_scores.npz and export_report_field_S1.json
